In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import sqlite3

In [9]:
orders = pd.read_csv("orders.csv")
users = pd.read_json("/content/drive/MyDrive/users.json")

In [10]:
orders.head()
users.head()

,user_id,name,city,membership
0,1,User_1,Chennai,Regular
1,2,User_2,Pune,Gold
2,3,User_3,Bangalore,Gold
3,4,User_4,Bangalore,Regular
4,5,User_5,Pune,Gold


In [15]:
conn  = sqlite3.connect("restaurants.db")
with open ("/content/drive/MyDrive/restaurants.sql", "r") as f:
  sql_script = f.read()
conn.executescript(sql_script)
restaurants = pd.read_sql("SELECT *FROM restaurants", conn)

In [16]:
restaurants.head()

,restaurant_id,restaurant_name,cuisine,rating
0,1,Restaurant_1,Chinese,4.8
1,2,Restaurant_2,Indian,4.1
2,3,Restaurant_3,Mexican,4.3
3,4,Restaurant_4,Chinese,4.1
4,5,Restaurant_5,Chinese,4.8


In [17]:
df = orders.merge(users,on="user_id", how="left")
df = df.merge(restaurants, on="restaurant_id", how="left")

In [18]:
df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           10000 non-null  int64  
 1   user_id            10000 non-null  int64  
 2   restaurant_id      10000 non-null  int64  
 3   order_date         10000 non-null  object 
 4   total_amount       10000 non-null  float64
 5   restaurant_name_x  10000 non-null  object 
 6   name               10000 non-null  object 
 7   city               10000 non-null  object 
 8   membership         10000 non-null  object 
 9   restaurant_name_y  10000 non-null  object 
 10  cuisine            10000 non-null  object 
 11  rating             10000 non-null  float64
dtypes: float64(2), int64(3), object(7)
memory usage: 937.6+ KB


In [19]:
df.to_csv("final_food_delivery_datasets.csv",index=False)

In [20]:
df[df['membership']=='Gold']\
.groupby('city')['total_amount']\
.sum()\
.sort_values(ascending=False)

,total_amount
city,
Chennai,1080909.79
Pune,1003012.32
Bangalore,994702.59
Hyderabad,896740.19


In [21]:
df.groupby('cuisine')['total_amount']\
.mean()\
.sort_values(ascending=False)

,total_amount
cuisine,
Mexican,808.021344
Italian,799.448578
Indian,798.466011
Chinese,798.389020


In [26]:
user_total = df.groupby('user_id')['total_amount'].sum()
count_users = user_total[user_total>1000].count()
count_users

np.int64(2544)

In [29]:
bins = [3.0,3.5,4.0,4.5,5.0]
labels = ['3.0-3.5', '3.6-4.0', '4.1-4.5','4.6-5.0']
df['rating_range'] = pd.cut(df['rating'], bins=bins, labels=labels,
include_lowest=True)

In [31]:
df.groupby('rating_range', observed=False)['total_amount'].sum().sort_values(ascending=False)

/tmp/ipython-input-678031030.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('rating_range')['total_amount'].sum().sort_values(ascending=False)


,total_amount
rating_range,
4.6-5.0,2197030.75
3.0-3.5,2136772.70
4.1-4.5,1960326.26
3.6-4.0,1717494.41


In [32]:
df[df['membership'] == 'Gold']\
.groupby('city')['total_amount']\
.mean()\
.sort_values(ascending=False)

,total_amount
city,
Chennai,808.459080
Hyderabad,806.421034
Bangalore,793.223756
Pune,781.162243


In [34]:
restaurant_count = df.groupby('cuisine')['restaurant_id'].nunique()

In [35]:
revenue = df.groupby('cuisine')['total_amount'].sum()

In [36]:
summary = pd.concat([restaurant_count, revenue], axis=1)
summary.columns = ['restaurant_count', 'revenue']
summary.sort_values(by=['restaurant_count', 'revenue'],ascending = [True,False])

,restaurant_count,revenue
cuisine,,
Chinese,120,1930504.65
Italian,126,2024203.80
Indian,126,1971412.58
Mexican,128,2085503.09


In [37]:
total_orders = df.shape[0]
gold_orders = df[df['membership'] == 'Gold'].shape[0]

percentage = round((gold_orders / total_orders) * 100)
percentage

50

In [39]:
restaurant_stats = df.groupby('restaurant_name_y').agg(
    total_orders=('order_id', 'count'),
    avg_order_value=('total_amount', 'mean')
)

restaurant_stats[restaurant_stats['total_orders'] < 20] \
.sort_values(by='avg_order_value', ascending=False)

,total_orders,avg_order_value
restaurant_name_y,,
Restaurant_294,13,1040.222308
Restaurant_262,18,1029.473333
Restaurant_77,12,1029.180833
Restaurant_193,15,1026.306667
Restaurant_7,16,1002.140625
...,...,...
Restaurant_184,19,621.828947
Restaurant_498,18,596.815556
Restaurant_192,14,589.972857


In [41]:
df[df['restaurant_name_y'] == 'Restaurant_294'][['restaurant_name_x', 'restaurant_name_y']].head()

,restaurant_name_x,restaurant_name_y
1407,Hotel Dhaba Multicuisine,Restaurant_294
1643,Hotel Dhaba Multicuisine,Restaurant_294
2426,Hotel Dhaba Multicuisine,Restaurant_294
3174,Hotel Dhaba Multicuisine,Restaurant_294
3243,Hotel Dhaba Multicuisine,Restaurant_294


In [45]:
df[df['restaurant_name_y'] == 'Restaurant_294']['restaurant_name_y'].unique()

array(['Restaurant_294'], dtype=object)

In [46]:
df.groupby(['membership', 'cuisine'])['total_amount'] \
.sum() \
.sort_values(ascending=False)

membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [47]:
df.groupby(['membership', 'cuisine'])['total_amount'] \
.sum() \
.sort_values(ascending=False)

membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [13]:
import os

print("Files in current working directory:")
print(os.listdir('.'))

Files in current working directory:
['.config', 'orders.csv', 'restaurants.db', 'drive', 'sample_data']


If `restaurants.sql` is not in the current directory, let's check your Google Drive (if it's mounted).

In [14]:
import os

drive_path = '/content/drive/MyDrive'

if os.path.exists(drive_path):
    print(f"Listing contents of {drive_path}:")
    # List all files and directories in MyDrive recursively
    for root, dirs, files in os.walk(drive_path):
        for file in files:
            if 'restaurants.sql' in file.lower(): # Case-insensitive search for restaurants.sql
                print(f"Found: {os.path.join(root, file)}")
else:
    print(f"Google Drive not mounted at {drive_path} or path does not exist.")
    print("Please ensure your Google Drive is mounted correctly.")

Listing contents of /content/drive/MyDrive:
Found: /content/drive/MyDrive/restaurants.sql


In [7]:
import os

drive_path = '/content/drive/MyDrive'

if os.path.exists(drive_path):
    print(f"Listing contents of {drive_path}:")
    # List all files and directories in MyDrive recursively
    for root, dirs, files in os.walk(drive_path):
        for file in files:
            if 'users.json' in file.lower(): # Case-insensitive search for users.json
                print(f"Found: {os.path.join(root, file)}")
else:
    print(f"Google Drive not mounted at {drive_path} or path does not exist.")
    print("Please ensure your Google Drive is mounted correctly.")

Listing contents of /content/drive/MyDrive:
Found: /content/drive/MyDrive/users.json


In [4]:
import os

print("Files in current working directory:")
print(os.listdir('.'))

Files in current working directory:
['.config', 'orders.csv', 'drive', 'sample_data']


If `users.json` is not in the current directory, let's check your Google Drive (if it's mounted).

In [5]:
import os

drive_path = '/content/drive/MyDrive'

if os.path.exists(drive_path):
    print(f"Listing contents of {drive_path}:")
    # List all files and directories in MyDrive recursively
    for root, dirs, files in os.walk(drive_path):
        for file in files:
            print(os.path.join(root, file))
else:
    print(f"Google Drive not mounted at {drive_path} or path does not exist.")
    print("Please ensure your Google Drive is mounted correctly.")

Listing contents of /content/drive/MyDrive:
/content/drive/MyDrive/harikakota-APSCHE_VLITS_Int-certificate-2.pdf
/content/drive/MyDrive/harikakota-APSCHE_VLITS_Int-certificate_copy.pdf
/content/drive/MyDrive/unit 4 questions and answers.pdf
/content/drive/MyDrive/FSD WEEK-1.docx
/content/drive/MyDrive/DBMS LAB MANUAL UPDATED R23.pdf
/content/drive/MyDrive/DBMS LAB MANUAL UPDATED R23.gdoc
/content/drive/MyDrive/nak.gdoc
/content/drive/MyDrive/Document from Harika (1).pdf
/content/drive/MyDrive/I am sharing 'microsoft internship' with you
/content/drive/MyDrive/CSP Registration Form.gdoc
/content/drive/MyDrive/CSP PROJECT.docx
/content/drive/MyDrive/20250731_220231.jpg
/content/drive/MyDrive/20250731_220614.jpg
/content/drive/MyDrive/20250731_221314.jpg
/content/drive/MyDrive/20250731_221705.jpg
/content/drive/MyDrive/Document from Harika.pdf
/content/drive/MyDrive/Document from Harika (2).gdoc
/content/drive/MyDrive/harika resume.docx
/content/drive/MyDrive/Document Harika.pdf
/content/

In [48]:
df['order_date'] = pd.to_datetime(df['order_date'])

df['quarter'] = df['order_date'].dt.to_period('Q')

df.groupby('quarter')['total_amount'] \
.sum() \
.sort_values(ascending=False)

/tmp/ipython-input-609749215.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['order_date'] = pd.to_datetime(df['order_date'])


,total_amount
quarter,
2023Q3,2037385.10
2023Q4,2018263.66
2023Q1,1993425.14
2023Q2,1945348.72
2024Q1,17201.50


In [49]:
df[df['membership'] == 'Gold']['order_id'].nunique()

4987

In [50]:
round(
    df[df['city'] == 'Hyderabad']['total_amount'].sum()
)

1889367

In [51]:
df['user_id'].nunique()

2883

In [52]:
round(
    df[df['membership'] == 'Gold']['total_amount'].mean(),
    2
)

np.float64(797.15)

In [53]:
df[df['rating'] >= 4.5]['order_id'].nunique()

3374

In [54]:
gold_city_revenue = (
    df[df['membership'] == 'Gold']
    .groupby('city')['total_amount']
    .sum()
    .sort_values(ascending=False)
)

gold_city_revenue

,total_amount
city,
Chennai,1080909.79
Pune,1003012.32
Bangalore,994702.59
Hyderabad,896740.19


In [55]:
top_city = gold_city_revenue.index[0]

df[
    (df['membership'] == 'Gold') &
    (df['city'] == top_city)
]['order_id'].nunique()

1337